In [1]:
import ipdb # <- трасировка и точки останова
import header
from header import __root__
from src import gs

🔑 Found password in password.txt (DEBUG MODE)
✅ Successfully opened KeePass database: C:\Users\user\Documents\repos\hypotez\secrets\credentials.kdbx
Failed to load GAPI credentials


In [2]:
import asyncio
from pathlib import Path
from types import SimpleNamespace
from typing import Optional, Dict, Any, List

from pydoll.browser.chrome import Chrome
from pydoll.constants import By

from src.llm.gemini import GoogleGenerativeAi # Unused, but kept
from src.endpoints.prestashop.product_fields import ProductFields

from src.endpoints.prestashop.product_async import PrestaProductAsync
from src.endpoints.prestashop.product import PrestaProduct

from src.utils.file import read_text_file, save_text_file, get_filenames_from_directory
from src.utils.jjson import j_loads, j_loads_ns, j_dumps # j_dumps unused
from src.utils.image import get_image_bytes, get_raw_image_data 
from src.utils.printer import pprint as print
from src.logger.logger import logger

In [3]:
class Config:
    """Класс конфигурации скрипта."""
    ENDPOINT: Path = __root__ / 'SANDBOX' / 'davidka'
    SUPPLIERS_ENDPOINT: Path = __root__ / 'src' / 'suppliers' / 'suppliers_list'
    SCENARIOS_DIR: Path = __root__ / 'SANDBOX' / 'davidka' / 'scenarios'
    # config: SimpleNamespace = j_loads_ns(ENDPOINT / 'davidka.json') #  general config.
    scenarios_files: List[str] = get_filenames_from_directory(SCENARIOS_DIR) # SANDBOX/davidka/scenarios/*.json
    PRESTA_API_KEY: str = gs.credentials.prestashop.store_davidka_net.api_key
    PRESTA_API_DOMAIN: str = gs.credentials.prestashop.store_davidka_net.api_domain
    #presta_product: PrestaProductAsync = PrestaProductAsync(api_key=PRESTA_API_KEY, api_domain=PRESTA_API_DOMAIN)

In [4]:
supplier_prefix:str = 'aliexpress'
supplier_alias:str = supplier_prefix.replace('.','_').replace('-','_')
supplier_config_path:Path = Config.SUPPLIERS_ENDPOINT / supplier_alias
locators_path:Path = supplier_config_path / 'locators'
product_locators:SimpleNamespace = j_loads_ns(locators_path / 'product.json')
category_locators:SimpleNamespace = j_loads_ns(locators_path / 'category.json')
product_url = fr'https://he.aliexpress_com.com/item/1005007819575751.html'
browser:Chrome = None
page:'Page' = None

In [5]:
async def run_scenario():
    """
    Исполнять сценарии лучше по такому шаблону"""
    
    async with Chrome() as browser:
        await browser.start()
        page = await browser.get_page()
        await page.go_to(product_url)
    

In [6]:
if not browser:
    browser = Chrome()  
    await browser.start()
    
if not page:
    page = await browser.get_page()
        

2025-06-09 15:46:40,234 - INFO - EventsHandler initialized
2025-06-09 15:46:40,240 - INFO - ConnectionHandler initialized.
2025-06-09 15:46:40,532 - INFO - Connecting to ws://localhost:9274/devtools/browser/10374313-8f60-4eed-a125-7186ea7c496b
2025-06-09 15:46:42,551 - INFO - EventsHandler initialized
2025-06-09 15:46:42,559 - INFO - ConnectionHandler initialized.


In [7]:
await page.go_to(product_url)

2025-06-09 15:46:42,581 - INFO - Connecting to ws://localhost:9274/devtools/page/9CD06C406825CC1363AB2CAF4209557F


In [8]:
# strategy: Dict[str, By] = {
#     'XPATH': By.XPATH,
#     'CSS_SELECTOR': By.CSS_SELECTOR,
# }

In [9]:
async def execute_locator(page,  locator: SimpleNamespace):
    """Locate and return content from the element based on locator info."""
    _webelement = await page.find_element(By[locator.by.upper()], locator.selector)
    #ipdb.set_trace()
    match locator.attribute.lower():
        case 'innertext':
            return await _webelement.get_element_text()
        case 'innerhtml':
            return await _webelement.inner_html
        case 'src':
            return _webelement.get_attribute('src')
    # Можно добавить return None или raise, если атрибут неизвестен

In [10]:
product_locators:SimpleNamespace = j_loads_ns(locators_path / 'product.json') # Обновить после редакции JSON 

In [11]:
f:ProductFields = ProductFields()

In [12]:
print(product_locators.name)

namespace(attribute='innerText', by='XPATH', selector="//h1[contains(@class,'product_title')] | //h1[@data-pl='product-title']", if_list='first', use_mouse=False, mandatory=True, timeout=0, timeout_for_event='presence_of_element_located', event=None, locator_description='name')


In [13]:
f.name = await execute_locator(page, product_locators.name)

In [14]:
f.price = await execute_locator(page, product_locators.price)

In [15]:
f.description =  await execute_locator(page, product_locators.description)

In [16]:
f.default_image_url =   await execute_locator(page, product_locators.default_image_url)

In [17]:
print(f.default_image_url)

https://ae-pic-a1.aliexpress-media.com/kf/S55481ad40e3648adb31f674b7e02715aV.jpg_220x220q75.jpg_.avif


In [18]:
#specification =  await execute_locator(page, product_locators.specification)

In [19]:
print(f.default_image_url)

https://ae-pic-a1.aliexpress-media.com/kf/S55481ad40e3648adb31f674b7e02715aV.jpg_220x220q75.jpg_.avif


In [20]:
# ПРАВИЛЬНО:
async with PrestaProductAsync(api_key=Config.PRESTA_API_KEY, api_domain=Config.PRESTA_API_DOMAIN) as presta_product_api:
    # Теперь presta_product_api.client инициализирован
    result = await presta_product_api.add_new_product_async(f)

2025-06-09 15:46:50,007 - DEBUG - 🐛 Попытка начального HEAD-запроса к https://store.davidka.net/api/ 
NoneType: None
2025-06-09 15:46:50,455 - INFO - HTTP Request: HEAD https://store.davidka.net/api/ "HTTP/1.1 200 OK"
2025-06-09 15:46:50,455 - INFO - ℹ️ Успешное подключение к PrestaShop. Версия: 9.0.0 
2025-06-09 15:46:50,462 - WARNING - ⚠️ Не удалось конвертировать начальный ID категории в int: None. Пропуск. 
2025-06-09 15:46:50,463 - DEBUG - 🐛 Начальные уникальные ID категорий (включая обработанные): {2} 
NoneType: None
2025-06-09 15:46:50,464 - WARNING - ⚠️ Не удалось конвертировать стартовый ID категории None в int. Пропуск ветки. 
2025-06-09 15:46:50,469 - DEBUG - 🐛 Финальный набор уникальных ID категорий: {2} 
NoneType: None
2025-06-09 15:46:50,469 - DEBUG - 🐛 Выполнение POST запроса к https://store.davidka.net/api/products с параметрами URL: {'output_format': 'JSON', 'language': 1} 
NoneType: None
2025-06-09 15:46:50,472 - ERROR - ❌ Неожиданная ошибка при вызове API к products 

PrestaShopException: 'Неожиданная ошибка: slice(None, 200, None)'

In [ ]:
fields = {
    'name': await page.find_element(strategy[locator_product.name.by], locator_product.name.selector),
    'price': await page.find_element(strategy[locator_product.price.by], locator_product.price.selector),
    'id_supplier': locator_product.id_supplier.attr, 
    'description_short': await page.find_element(strategy[locator_product.description_short.by], locator_product.description_short.selector),
    'description': await page.find_element(strategy[locator_product.description.by], locator_product.description.selector),
    'specification': await page.find_element(strategy[locator_product.specification.by], locator_product.specification.selector),
    'default_image_url': await page.find_element(strategy[locator_product.default_image_url.by], locator_product.default_image_url.selector),
}